# FollowBench-MK — model inference

This notebook runs open-weight language models on the FollowBench-MK prompts and saves
their answers. The prompts are the English FollowBench instructions (levels 1–5) and their
Macedonian translations.

Each answer is written to Google Drive as soon as it is produced. If the Colab session
stops, running the notebook again continues from the last saved answer instead of starting
over.

The work can be split across several Google accounts: each account runs a different model
(see "Which models to run"). Results are stored in one file per model and language, so
parallel runs never write to the same file.

## 1. Install and import

In [ ]:
!nvidia-smi

Only standard libraries are used: `transformers` for the models, plus `torch`, `pandas`, and the Python standard library.

In [ ]:
!pip install -q transformers==5.18.0 accelerate bitsandbytes

In [ ]:
import os, json, time
from datetime import datetime, timezone

import torch
import pandas as pd
from huggingface_hub import login
from transformers import (AutoTokenizer, AutoModelForCausalLM,
                          AutoProcessor, AutoModelForMultimodalLM, BitsAndBytesConfig)

## 2. Which models to run

`MODELS_TO_RUN` selects the models this session will generate answers for. To run the
experiment from several accounts at the same time, each account keeps a different line
uncommented. Because every model writes to its own result file, the accounts can run in
parallel and the files can be merged afterwards.

An account that finishes early can add an unfinished model to its list and run again; any
model and prompt that is already complete is skipped.

In [ ]:
# One line per account; the four models were run in parallel.
MODELS_TO_RUN = ['qwen3.5-9b']
# MODELS_TO_RUN = ['gemma-4-12b']
# MODELS_TO_RUN = ['llama-3.1-8b']
# MODELS_TO_RUN = ['domestic-yak-8b']

## 3. Settings

Generation settings are the same for every model so the answers are comparable. Greedy
decoding (`do_sample=False`) makes a run repeatable. `LIMIT` is used for a quick test on a
few prompts; `None` runs all of them.

In [ ]:
MAX_NEW_TOKENS = 2048
SEED = 42
LANGUAGES = ['mk', 'en']
LIMIT = None

# 4-bit loading lets the ~8-12B models fit on a 16 GB GPU (e.g. T4).
# On a larger GPU (L4/A100) this can be set to False to load in bf16.
USE_4BIT = True

Each model and the exact revision that produced the answers. Qwen 3.5 and Gemma 4 are
multimodal checkpoints and load through a processor; only text is sent to them. The
Llama 3.1 revision was not pinned (`None`), so the latest version at the time of the run
was used.

In [ ]:
MODELS = {
    'domestic-yak-8b': {'id': 'LVSTCK/domestic-yak-8B-instruct',
                        'revision': '877f355e049724338b3ae831ac117252391d8f91', 'kind': 'text'},
    'llama-3.1-8b':    {'id': 'meta-llama/Llama-3.1-8B-Instruct',
                        'revision': None, 'kind': 'text'},
    'qwen3.5-9b':      {'id': 'Qwen/Qwen3.5-9B',
                        'revision': 'c202236235762e1c871ad0ccb60c8ee5ba337b9a', 'kind': 'multimodal'},
    'gemma-4-12b':     {'id': 'google/gemma-4-12B-it',
                        'revision': '707f0a3b8a3c7ad586ed01e27eafbad8a27dd0f7', 'kind': 'multimodal'},
}

## 4. GPU check

Prints the GPU and its memory. Useful to record which hardware produced the answers.

In [ ]:
print('GPU:', torch.cuda.get_device_name(0))
print('Memory: %.1f GB' % (torch.cuda.get_device_properties(0).total_memory / 1024**3))

## 5. Google Drive

Drive holds the prompts and the results, so they survive a disconnect. Copy the two prompt
files from `data/prompts/` (`followbench_en.jsonl` and `followbench_mk.jsonl`) into
`DRIVE_DIR` before running. Results are written to the `results` subfolder; these are the
files in `data/responses/raw/`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/followbench_mk'
RESULTS_DIR = DRIVE_DIR + '/results'
os.makedirs(RESULTS_DIR, exist_ok=True)

print('files in', DRIVE_DIR, ':', os.listdir(DRIVE_DIR))

## 6. Hugging Face login

Required for the gated Llama 3.1 download. The token is stored in Colab secrets (the key icon) under the name `HF_TOKEN`.

In [ ]:
from google.colab import userdata
login(token=userdata.get('HF_TOKEN'))

## 7. Load the prompts

Both language files are loaded and combined into one list. Each record has: `pilot_id`,
`family_category`, `level`, `lang`, and the `instruction` text that is sent to the model.

In [ ]:
def load_jsonl(path):
    rows = []
    with open(path, encoding='utf-8') as f:
        for line in f:
            if line.strip():
                rows.append(json.loads(line))
    return rows

prompts = []
for lang in LANGUAGES:
    for row in load_jsonl('%s/followbench_%s.jsonl' % (DRIVE_DIR, lang)):
        row['lang'] = lang
        prompts.append(row)

print(len(prompts), 'prompts loaded')
pd.DataFrame(prompts).groupby(['lang', 'category_group']).size()

## 8. Result files and resume

Each model and language has its own result file. Before a model runs, the ids already in
its file are read so they can be skipped. This is what lets a run continue after a
disconnect.

In [ ]:
def result_path(model_name, lang):
    return '%s/%s__%s.jsonl' % (RESULTS_DIR, model_name, lang)

def already_done(path):
    if not os.path.exists(path):
        return set()
    return {json.loads(line)['pilot_id'] for line in open(path, encoding='utf-8') if line.strip()}

One answer is appended per line. The file is flushed to disk immediately so nothing is lost if the session stops.

In [ ]:
def save_answer(path, record):
    with open(path, 'a', encoding='utf-8') as f:
        f.write(json.dumps(record, ensure_ascii=False) + '\n')
        f.flush()
        os.fsync(f.fileno())

## 9. Load a model

Text models load with a tokenizer, multimodal models with a processor. 4-bit loading
(NF4) keeps memory low enough for a 16 GB GPU. The resolved revision is returned so it can
be saved with every answer.

In [ ]:
def load_model(model_name):
    info = MODELS[model_name]
    print('loading', info['id'])

    args = {'device_map': 'cuda'}
    if USE_4BIT:
        args['quantization_config'] = BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_quant_type='nf4', bnb_4bit_use_double_quant=True,
            bnb_4bit_compute_dtype=torch.float16)
    else:
        args['dtype'] = torch.bfloat16

    if info['kind'] == 'multimodal':
        proc = AutoProcessor.from_pretrained(info['id'], revision=info['revision'])
        model = AutoModelForMultimodalLM.from_pretrained(info['id'], revision=info['revision'], **args)
    else:
        proc = AutoTokenizer.from_pretrained(info['id'], revision=info['revision'])
        model = AutoModelForCausalLM.from_pretrained(info['id'], revision=info['revision'], **args)
        if proc.pad_token_id is None:
            proc.pad_token = proc.eos_token

    model.eval()
    print('revision:', info['revision'])
    return proc, model

## 10. Generate one answer

The prompt is wrapped in the model's own chat template as a single user turn, with no
system prompt. `status` records whether the model stopped on its own (`ok`) or hit the
token limit (`truncated`).

In [ ]:
def build_inputs(proc, kind, text):
    if kind == 'multimodal':
        messages = [{'role': 'user', 'content': [{'type': 'text', 'text': text}]}]
        inputs = proc.apply_chat_template(messages, add_generation_prompt=True, tokenize=True,
                                          return_dict=True, return_tensors='pt', enable_thinking=False)
    else:
        messages = [{'role': 'user', 'content': text}]
        inputs = proc.apply_chat_template(messages, add_generation_prompt=True, tokenize=True,
                                          return_dict=True, return_tensors='pt')
    return inputs.to('cuda')

In [ ]:
def generate(proc, model, kind, text):
    inputs = build_inputs(proc, kind, text)
    n_in = inputs['input_ids'].shape[1]
    tok = getattr(proc, 'tokenizer', proc)
    pad_id = tok.pad_token_id if tok.pad_token_id is not None else tok.eos_token_id

    start = time.time()
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False, pad_token_id=pad_id)
    seconds = round(time.time() - start, 1)

    new_tokens = out[0][n_in:]
    answer = tok.decode(new_tokens, skip_special_tokens=True).strip()
    status = 'truncated' if len(new_tokens) >= MAX_NEW_TOKENS else 'ok'
    return answer, len(new_tokens), seconds, status

## 11. Run

For each selected model: load it once, go through the prompts that are not yet saved (both
languages), and write each answer immediately. Models already finished are skipped, so the
cell can be re-run safely after a disconnect.

In [ ]:
def run_model(model_name):
    todo = {}
    for lang in LANGUAGES:
        done = already_done(result_path(model_name, lang))
        todo[lang] = [p for p in prompts if p['lang'] == lang and p['pilot_id'] not in done]

    if sum(len(v) for v in todo.values()) == 0:
        print(model_name, '- already complete, skipping')
        return

    torch.manual_seed(SEED)
    proc, model = load_model(model_name)

    for lang in LANGUAGES:
        items = todo[lang][:LIMIT] if LIMIT else todo[lang]
        print('%s / %s : %d to generate' % (model_name, lang, len(items)))
        for i, p in enumerate(items, 1):
            answer, n_tokens, seconds, status = generate(proc, model, MODELS[model_name]['kind'], p['instruction'])
            save_answer(result_path(model_name, lang), {
                'model': model_name, 'model_id': MODELS[model_name]['id'],
                'revision': MODELS[model_name]['revision'], 'lang': lang,
                'pilot_id': p['pilot_id'], 'family_category': p['category_group'], 'level': p['level'],
                'answer': answer, 'output_tokens': n_tokens, 'seconds': seconds, 'status': status,
                'gpu': torch.cuda.get_device_name(0), 'use_4bit': USE_4BIT,
                'time_utc': datetime.now(timezone.utc).isoformat(timespec='seconds'),
            })
            print('  [%d/%d] %s  %d tok  %.1fs  %s' % (i, len(items), p['pilot_id'], n_tokens, seconds, status))

    del model, proc
    torch.cuda.empty_cache()
    print(model_name, '- done')

In [ ]:
for model_name in MODELS_TO_RUN:
    run_model(model_name)

## 12. Progress

Loads every result file and shows how many answers are saved per model and language, with truncation counts and total time.

In [ ]:
results = []
for fname in sorted(os.listdir(RESULTS_DIR)):
    if fname.endswith('.jsonl'):
        results += load_jsonl(RESULTS_DIR + '/' + fname)

if results:
    rdf = pd.DataFrame(results)
    summary = rdf.groupby(['model', 'lang']).agg(
        answers=('pilot_id', 'count'),
        truncated=('status', lambda s: (s == 'truncated').sum()),
        minutes=('seconds', lambda s: round(s.sum() / 60, 1)))
    print(summary)
    print('\nduplicates:', rdf.duplicated(['model', 'lang', 'pilot_id']).sum())
else:
    print('no results yet')

## 13. Read some answers

Shows all saved answers for one prompt. The `pilot_id` can be changed to inspect others.

In [ ]:
pilot_id = 'situation_008_L3'

view = pd.DataFrame(results)
view = view[view['pilot_id'] == pilot_id][['model', 'lang', 'status', 'answer']]
for _, r in view.iterrows():
    print('---', r['model'], r['lang'], '(%s)' % r['status'])
    print(r['answer'][:800])
    print()